In [ ]:
import json
import shapely
import geopandas as gpd
import numpy as np

from theia.openburst_client import OpenburstClient, PclReceiver, LatLonHeightGrid
from theia.types import Point, Radar, Polarization

# Client library

In [ ]:
def coverage_polygon_to_mask(
    polygon: shapely.geometry.polygon.Polygon,
    center_lon: float,
    center_lat: float,
    extent_lon: float,
    extent_lat: float,
    width: int,
    height: int,
) -> np.ndarray:
    # Calculate region represented by the mask.
    lon_min = center_lon - 0.5 * extent_lon
    lon_max = center_lon + 0.5 * extent_lon

    lat_min = center_lat - 0.5 * extent_lat
    lat_max = center_lat + 0.5 * extent_lat

    # Clip overlay to bbox.
    bbox = gpd.GeoDataFrame(
        geometry=[
            gpd.points_from_xy([lon_min, lon_max], [lat_min, lat_max])
            .union_all()
            .envelope
        ],
        crs="EPSG:4326",
    )
    df = gpd.GeoDataFrame([], geometry=[polygon])
    df.set_crs("EPSG:4326", inplace=True)
    overlay_clipped = df.clip(bbox)

    # Raster transform.
    transform = from_bounds(lon_min, lat_min, lon_max, lat_max, width, height)

    # Rasterize to create binary mask.
    mask = rasterize(
        [(geom, 1) for geom in overlay_clipped.geometry],
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype=np.uint8,
    )
    return mask

# Config

In [ ]:
transmitter = Radar(
    id=585,
    point=Point(
        lat=47.36700085728634,
        lon=8.537724304199216,
        alt=407.83600886023686,
    ),
    power=20000,
    erp=800,
    antenna_height=10.0,
    diameter=2.0,
    frequency=1000.0,
    pulse_width=1,
    cpi_pulses=1,
    bandwidth=1,
    pfa=1e-6,
    min_elevation=-20.0,
    max_elevation=60.0,
    rotation_time=10.0,
    polarization=Polarization.HORIZONTAL,
)

receiver = transmitter.model_copy()
receiver.point.lat = 47.28968537077668
receiver.point.lon = 8.608019638061526
receiver.point.alt = 160.6

target_flight_height: float = 1000.0
target_cross_section: float = 2.0

# Parameters for controlling the resolution of the binary mask.
# Mask extent in degrees.
extent = 0.6

# Mask resolution in pixels.
width, height = 1024, 1024

# Calculate Coverage

In [ ]:
client = OpenburstClient("localhost")
coverage = client.calculate_coverage(
    transmitter,
    target_flight_height,
    target_cross_section,
)

In [ ]:
lon, lat = coverage.exterior.coords.xy
with open("reference.geosjon", "w") as file:
    json.dump(
        {
            "type": "Polygon",
            "coordinates": [np.stack([lon, lat], axis=1).tolist()],
        },
        file,
    )

# Plot

In [ ]:
import folium

In [ ]:
m = folium.Map(location=(46.7, 10.3), zoom_start=8)
overlay = folium.GeoJson(coverage).add_to(m)

m.save("map.html")

# Rendering to image

mask = coverage_polygon_to_mask(
    coverage, transmitter.lon, transmitter.lat, extent, extent, width, height
)

import matplotlib.pyplot as plt

plt.imshow(mask, aspect=1.0)

# Query elevation

In [ ]:
client = OpenburstClient("localhost")
client.elevation_at(47.38301680962604, 8.495104310332978)

# Calculate Propation

In [ ]:
client = OpenburstClient("localhost")
result = client.calculate_propagation(transmitter, receiver)
result

In [ ]:
%%timeit
result = client.calculate_propagation(transmitter, receiver)

# PCL

In [ ]:
receivers = [
    PclReceiver(
        name="PCL_SENSOR_0",
        rx_id=1,
        lat=46.95556964077278,
        lon=8.638317871093756,
        masl=1151,
    ),
    PclReceiver(
        name="PCL_SENSOR_1",
        rx_id=2,
        lat=46.99679762141241,
        lon=8.881390380859381,
        masl=1844,
    ),
]

In [ ]:
client = OpenburstClient("localhost")
# Get "Emitters of opportunity".
pcl_receivers, pcl_emitters = client.get_pcl_receivers_emitters(receivers)

In [ ]:
import pandas as pd

In [ ]:
pd.DataFrame([r.model_dump() for r in pcl_receivers]).set_index("rx_id")

In [ ]:
pd.DataFrame([e.model_dump() for e in pcl_emitters]).set_index("tx_id")

In [ ]:
client = OpenburstClient("localhost")
# Get "Emitters of opportunity".
pcl_receivers, pcl_emitters = client.get_pcl_receivers_emitters(receivers)

grid = LatLonHeightGrid(
    lat_start=46.85611714645961,
    lat_stop=46.99679762141241,
    lat_res=0.015631163883644678,
    lon_start=8.300488281249999,
    lon_stop=8.881390380859381,
    lon_res=0.06454467773437579,
    height_start=3000.0,
    height_stop=3000.0,
    height_res=3000.0,
)

In [ ]:
cubes_per_receiver = client.calculate_min_det_rcs_coverage(
    pcl_receivers, pcl_emitters, grid
)

In [ ]:
cubes_per_receiver[1]

In [ ]:
import matplotlib.pyplot as plt
import geojsoncontour

fig, ax = plt.subplots()

data = cubes_per_receiver[0]

contour = ax.contourf(
    np.linspace(grid.lon_start, grid.lon_stop, grid.n_points_lon),
    np.linspace(grid.lat_start, grid.lat_stop, grid.n_points_lat),
    data[:, :, 0],
)
geojson = geojsoncontour.contourf_to_geojson(contourf=contour, ndigits=3, unit="m")
plt.close(fig)

import folium
from folium.plugins import HeatMap

m = folium.Map(location=(grid.center[0], grid.center[1]))
for receiver in pcl_receivers:
    folium.Marker(
        location=(receiver.lat, receiver.lon),
        tooltip=receiver.name,
        icon=folium.Icon(icon="ear-listen", prefix="fa", color="blue"),
    ).add_to(m)
for emitter in pcl_emitters:
    folium.Marker(
        location=(emitter.lat, emitter.lon),
        tooltip=emitter.callsign,
        icon=folium.Icon(icon="tower-cell", prefix="fa", color="blue"),
    ).add_to(m)
folium.GeoJson(geojson).add_to(m)
m.save("map.html")

# Active radar detections

In [ ]:
from theia.openburst_client import Target

transmitter = Radar(
    id=585,
    point=Point(
        lat=47.36700085728634,
        lon=8.537724304199216,
        alt=407.83600886023686,
    ),
    power=20000,
    erp=800,
    antenna_height=10.0,
    diameter=2.0,
    frequency=1000.0,
    pulse_width=1,
    cpi_pulses=1,
    bandwidth=1,
    pfa=1e-6,
    min_elevation=-20.0,
    max_elevation=60.0,
    rotation_time=10.0,
    polarization=Polarization.HORIZONTAL,
)

target = Target(
    id=0,
    lat=47.348,
    lon=8.6266,
    alt=1000.0,
    cross_section=1.0,
    vlon=250.0,
    vlat=0.0,
    vz=0.0,
)

target2 = Target(
    id=2,
    lat=47.406,
    lon=8.3926,
    alt=1000.0,
    cross_section=1.0,
    vlon=250.0,
    vlat=0.0,
    vz=0.0,
)

receiver = transmitter.model_copy()
receiver.id = 1
receiver.point.lat = target.lat
receiver.point.lon = target.lon
receiver.point.alt = target.alt

receiver2 = transmitter.model_copy()
receiver2.id = 2
receiver2.point.lat = target2.lat
receiver2.point.lon = target2.lon
receiver2.point.alt = target2.alt

In [ ]:
client.active_radar_probability_of_detection(transmitter, target)

In [ ]:
client.active_radar_probability_of_detection(transmitter, target2)

In [ ]:
client = OpenburstClient("localhost")
result = client.calculate_propagation(transmitter, receiver2)
result[0]

In [ ]:
polygon = contour.get_paths()[0].to_polygons()[0]  # [[0, -1]]

In [ ]:
type(contour.get_paths()[0])

In [ ]:
import matplotlib

def polygon_to_geojson(polygon: np.ndarray) -> str:
    """
    Parameters
    ----------
    polygon: np.ndarray
        Array of shape (N, 2) containing (lon, lat) coordinates as rows.
    """
    return {
        "type": "feature",
        "geometry": {
            "type": "Polygon",
            "coordinates": polygon.tolist(),
        }
    }


def path_to_geojson(path: matplotlib.path.Path) -> str:


polygon_to_geojson(polygon)

# Load replay data

In [ ]:
import pandas as pd

cols = [
    "DateTimeIndex",
    "millisecs",
    "converted_integer_id",
    "lat",
    "lon",
    "heading[0 = north\n180 = south\n360 = north]",
    "speed [km / h]",
    "altitude[m]",
    "track_quality",
    "milli_secs_after_midnight",
    "tgt_vx [vel m/s on lon axis]",
    "tgt_vy [vel m/s on lat axis]",
    "tgt_vz [vel m/s on z axis]",
]

df = pd.DataFrame(
    np.load(
        "./openburst/replay/DATA/eastbound_single_4000masl.npy",
        allow_pickle=True,
    ),
    columns=cols,
)

# Only one target.
assert len(df["converted_integer_id"].unique()) == 1

path = {
    "type": "LineString",
    "coordinates": df[["lon", "lat"]].values.tolist(),
}

In [ ]:
map = folium.Map()

folium.GeoJson(path).add_to(map)
map.save("map.html")

In [ ]:
delta_t = df["DateTimeIndex"] - df["DateTimeIndex"].iloc[0]

In [ ]:
delta_t.dt.microseconds

In [ ]:
df